In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader
from dataset import PendulumDataset
from model import LSTMModel
import torch.optim as optim

torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
data_paths = []
dataset = PendulumDataset(data_paths)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

batch_size = 50
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

input_size = 4
hidden_layer_size = 50 
output_size = 4
num_layers = 3
learning_rate = 0.001
num_epoch = 100 

model = LSTMModel(input_size, hidden_layer_size, num_layers, output_size).to(device)
loss_function = nn.MSELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def train_model(model, train_loader, test_loader, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
    
        for sequences, targets in train_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            batch_size, seq_len, input_dim = sequences.size()
            
            optimizer.zero_grad()
            output_t = model(sequences) 
            loss = loss_function(output, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

def teacher_forcing(model, train_loader, test_loader, num_epochs, pred_len):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
    
        for sequences, targets in train_loader:
            sequences, targets = sequences.to(device), targets.to(device)
            batch_size, seq_len, input_dim = sequences.size()
            optimizer.zero_grad()
            
            outputs = torch.zeros(batch_size, pred_len, 4).to(device)
            input_t = sequences[:, -1, :] 
            
            for t in range(pred_len):
                output_t, _ = model(input_t.unsqueeze(1))
                outputs[:, t, :] = output_t
                input_t = targets[:, t, :]
                
            loss = loss_function(output, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        

ModuleNotFoundError: No module named 'torch'